# Expedia Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/expedia_reviews.py`.

Covers:
1. Web scraping – fetching reviews from Expedia hotel pages
2. Ollama classification – topic & sentiment detection
3. Manual review input – adding reviews that the scraper can't fetch

**Requirements:** Ollama running locally with `mistral:7b`.
No API key needed – Expedia reviews are scraped from the public website.

**Note:** Expedia has anti-bot measures. The scraper uses URL candidates,
UA rotation, and retry logic to work around rate limits.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [2]:
from src.sites import expedia_reviews as expr

hotel_url = expr.EXPEDIA_URLS.get(expr.ANANEA_HOTEL)
base_url = expr._hotel_url_to_base_url(hotel_url)
print(f'Hotel: {expr.ANANEA_HOTEL}')
print(f'Hotel URL: {hotel_url}')
print(f'Base URL: {base_url}')

Hotel: Ananea Castelo Suites Hotel
Hotel URL: https://euro.expedia.net/Albufeira-Hotels-Castelo-Suites-Hotel.h111521689.Hotel-Information?pwaDialog=product-reviews
Base URL: https://euro.expedia.net/Albufeira-Hotels-Castelo-Suites-Hotel.h111521689.Hotel-Information


## 1. Web Scraping – Single Page

In [3]:
# Fetch reviews from Expedia using Playwright (clicks "See all reviews" button)
html = expr.fetch_reviews_page(hotel_url, timeout=30000)
if html:
    print(f'Content length: {len(html)} chars')
    page_reviews = expr._scrape_reviews_from_html(html)
    print(f'Reviews found on page: {len(page_reviews)}')
    print()
    for r in page_reviews:
        rating = r.get('rating') or '?'
        print(f"  [{r.get('id', 'N/A')[:16]}] {rating}/10 – {r.get('title', '(no title)')[:50]} ({r.get('travel_date', '')})")
else:
    print('Failed to fetch page (anti-bot block?)')

Content length: 1483059 chars
Reviews found on page: 8

  [68e299f462a90e06] 8.0/10 – Good (2025-10-06)
  [68b68904eb5adc43] 10.0/10 – Excellent (2025-09-02)
  [689eee3603ad6a6f] 8.0/10 – Good (2025-08-17)
  [68e66af970871a13] 6.0/10 – Okay (2025-10-08)
  [68dee7999267ef09] 10.0/10 – Excellent (2025-10-03)
  [68d989ceac02a713] 10.0/10 – Excellent (2025-09-28)
  [68c80408780f2428] 10.0/10 – Excellent (2025-09-15)
  [68ab575066fec64f] 8.0/10 – Good (2025-08-24)


## 2. Web Scraping – All Pages with Pagination

In [4]:
# Fetch reviews across multiple pages
all_reviews = expr.expedia_get_reviews(hotel_url, max_pages=3, min_delay=2.0, max_delay=4.0)
print(f'Total unique reviews fetched: {len(all_reviews)}')
print()
for r in all_reviews:
    rating = r.get('rating') or '?'
    print(f"  [{r.get('id', 'N/A')[:16]}] {rating}/10 \u2013 {r.get('title', '(no title)')[:50]} ({r.get('travel_date', '')})")

Total unique reviews fetched: 8

  [68e299f462a90e06] 8.0/10 – Good (2025-10-06)
  [68b68904eb5adc43] 10.0/10 – Excellent (2025-09-02)
  [689eee3603ad6a6f] 8.0/10 – Good (2025-08-17)
  [68e66af970871a13] 6.0/10 – Okay (2025-10-08)
  [68dee7999267ef09] 10.0/10 – Excellent (2025-10-03)
  [68d989ceac02a713] 10.0/10 – Excellent (2025-09-28)
  [68c80408780f2428] 10.0/10 – Excellent (2025-09-15)
  [68ab575066fec64f] 8.0/10 – Good (2025-08-24)


In [5]:
# Inspect raw scraped data for first review
if all_reviews:
    print(json.dumps(all_reviews[0], indent=2, ensure_ascii=False))

{
  "id": "68e299f462a90e06c39e2b5e",
  "rating": 8.0,
  "title": "Good",
  "text": "Breakfast was lovely i do wish the room had more storage. Lovely location.",
  "travel_date": "2025-10-06",
  "author_name": "Michelle"
}


## 3. Ollama Classification – Test

In [6]:
# Check Ollama availability
import requests
from src.classification import classify_review, is_ollama_available

ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')

Ollama available: True
Models: ['mistral:7b']


In [7]:
# Classify a single review (pick the first with text)
test_review = next((r for r in all_reviews if r.get('text')), None)
if test_review and ollama_ok:
    print(f"Review: {test_review.get('title', '(no title)')}")
    print(f"Text: {test_review['text'][:300]}...")
    print()
    topics = classify_review(test_review['text'])
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('No review with text found or Ollama not available')

Review: Good
Text: Breakfast was lovely i do wish the room had more storage. Lovely location....

Classification result (2 topics):
  🟢 meals = positive
  🔴 commodities = negative


In [ ]:
# Classify ALL fetched reviews, save to JSON
from datetime import datetime

if ollama_ok:
    json_path = str(ROOT / 'data' / 'expedia_reviews.json')
    existing = expr.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in all_reviews:
        text = r.get('text', '')
        if not text:
            continue
        topics = classify_review(text)

        review_id = str(r.get('id', ''))

        record = {
            'id': review_id,
            'hotel': expr.ANANEA_HOTEL,
            'source': 'expedia',
            'rating': r.get('rating'),
            'title': r.get('title', ''),
            'text': text,
            'published_date': r.get('travel_date', ''),
            'author_name': r.get('author_name', ''),
            'country': r.get('country', '') or 'Unknown',
            'trip_type': r.get('trip_type', '') or 'Unknown',
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if review_id in existing_by_id else 'added'
        existing_by_id[review_id] = record

        rating = r.get('rating') or 0
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{rating}/10 [{action}] {r.get('title', 'N/A')[:45]:45s} \u2192 {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    expr.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available \u2013 skip classification')

## 4. Manual Review Input

Expedia pages may block scraping or return limited results.
Use this section to manually add reviews you found on the website.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'reviewer_name': 'Guest',                       # reviewer name (used for ID)
    'rating': 8.0,                                  # 0-10 (Expedia scale)
    'title': 'Great hotel',                         # review title
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
    'trip_type': 'Couple',                          # Couple, Family, Friends, Solo, Business
    'country': 'Unknown',                           # reviewer country (or Unknown)
}
# ========================================

# Generate unique ID from name + date + title
id_seed = f"{manual_review['reviewer_name']}_{manual_review['published_date']}_{manual_review['title']}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

record = {
    'id': review_id,
    'hotel': expr.ANANEA_HOTEL,
    'source': 'expedia',
    'rating': manual_review['rating'],
    'title': manual_review['title'],
    'text': manual_review['text'],
    'published_date': manual_review['published_date'],
    'author_name': manual_review['reviewer_name'],
    'country': manual_review.get('country', '') or 'Unknown',
    'trip_type': manual_review.get('trip_type', '') or 'Unknown',
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = classify_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text \u2013 skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'expedia_reviews.json')
existing = expr.load_reviews(json_path)

existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'\u26a0\ufe0f  Review {record["id"]} already exists \u2013 skipping')
else:
    existing.append(record)
    expr.save_reviews(existing, json_path)
    print(f'\u2705 Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once.

In [ ]:
import hashlib

batch_reviews = [
    {
        'reviewer_name': 'Traveler',
        'rating': 9.0,
        'title': 'Wonderful stay',
        'text': 'Replace with actual review text...',
        'published_date': '2026-02-15',
        'trip_type': 'Couple',
        'country': 'Unknown',
    },
    # Add more reviews here...
]

json_path = str(ROOT / 'data' / 'expedia_reviews.json')
existing = expr.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['reviewer_name']}_{mr['published_date']}_{mr['title']}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

    if rid in existing_ids:
        print(f'\u26a0\ufe0f  Skip duplicate: {mr["title"]}')
        continue

    rec = {
        'id': rid,
        'hotel': expr.ANANEA_HOTEL,
        'source': 'expedia',
        'rating': mr['rating'],
        'title': mr['title'],
        'text': mr['text'],
        'published_date': mr.get('published_date', ''),
        'author_name': mr['reviewer_name'],
        'country': mr.get('country', '') or 'Unknown',
        'trip_type': mr.get('trip_type', '') or 'Unknown',
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
    }

    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = classify_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')

    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"\u2705 {mr['title']} \u2192 {topic_str or '(unclassified)'}")

if added:
    expr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [ ]:
json_path = str(ROOT / 'data' / 'expedia_reviews.json')
reviews = expr.load_reviews(json_path)

needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            print(f"  \u2705 {r.get('title', 'N/A')[:40]} \u2192 {topic_str or '(still none)'}")
        except Exception as e:
            print(f"  \u274c {r.get('title', 'N/A')[:40]} \u2192 Error: {e}")

    expr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

## 7. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [ ]:
json_path = str(ROOT / 'data' / 'expedia_reviews.json')
reviews = expr.load_reviews(json_path)

with_text = [r for r in reviews if r.get('text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_review(r['text'])
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '\U0001f504' if old_str != new_str else '\u2705'
            print(f"  {changed} [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            print(f"  \u274c [{i}/{len(with_text)}] {r.get('title', 'N/A')[:40]} \u2192 Error: {e}")

    expr.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

## 8. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'expedia_reviews.json')
reviews = expr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if "manual" in str(r.get("id", "")))}')
print()

# Rating distribution (0-10 scale)
ratings = [r.get('rating') for r in reviews if r.get('rating') is not None]
if ratings:
    print(f'Average rating: {sum(ratings)/len(ratings):.1f}/10')
    print(f'Rating range: {min(ratings):.1f} - {max(ratings):.1f}')
    print()

# Country breakdown
country_counts = {}
for r in reviews:
    c = r.get('country', 'Unknown') or 'Unknown'
    country_counts[c] = country_counts.get(c, 0) + 1
if country_counts:
    print('Country breakdown:')
    for c, n in sorted(country_counts.items(), key=lambda x: -x[1]):
        print(f'  {c}: {n}')
    print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')